In [270]:
import os
from .storage import Page

class Pager:


# Accept a file_path.

# If the file doesn’t exist:

# Create it using open(..., 'w+b').

# Open the file using open(..., 'r+b') and store that file object in self.file.

# Set a page_size of 4096.

# Create an empty dictionary self.cache = {} to store in-memory pages.

# Set self.file_size by calling os.path.getsize(file_path).

# Calculate self.page_count = file_size // page_size.

        def __init__(self, file_path:str):
                # Does the file exist? Create with writing priviliges if not, or reading priviliges if yes
                if not os.path.exists(file_path): 
                       self.file = open(file_path,'w+b') 
                else:
                       self.file = open(file_path,'r+b') 
                self.page_size = 4096
                self.cache = {}
                self.file_size = os.path.getsize(file_path)
                self.page_count = self.file_size // self.page_size

        def get_page(self, page_id):
                if page_id in self.cache: # if already in cache, return the page
                        return self.cache[page_id]
                else: # if not read the page from disc
                        offset = self.page_size * page_id #since we are storing pages in order, the id of a page denotes its position in memory
                        self.file.seek(offset) # move to the location where the page starts
                        raw_data = self.file.read(self.page_size)
                        page = Page.from_bytes(raw_data) # create a page from the memory you just read in
                        self.cache[page_id]=page # store the page in the cache
                        return page
                
        def flush_page(self,page_id):
                if page_id not in self.cache: #if page already not in cache then exit the function
                        return
                page = self.cache[page_id] # retrieve the in-memory page to persist it
                page_bytes = page.to_bytes() #serialize the page and return it as a new variable in bytes format
                offset = self.page_size * page_id #find the correct location to store this page
                self.file.seek(offset) # move to the location on disc where the page needs to start
                self.file.write(page_bytes) #  write the raw data to this location of the file
                self.file.flush() # flush internal write buffer immediately


        def flush_all(self):
                for page_id, page in self.cache.items():
                        page_bytes = page.to_bytes() #serialize the page and return it as a new variable in bytes format
                        offset = self.page_size * page_id #find the correct location to store this page
                        self.file.seek(offset) # move to the location on disc where the page needs to start
                        self.file.write(page_bytes) #  write the raw data to this location of the file
                self.file.flush() # flush internal write buffer immediately after loop ends
        
                        
                        
                        
            
                

ImportError: attempted relative import with no known parent package

In [256]:
dic = {1:'first',2:'second',3:'third'}

In [267]:
dic.values()

dict_values(['first', 'second', 'third'])

In [225]:
class Page:
    # - memory counter how much memory does this page hold right now
    # - maximum memory per page (we want 4096 Bytes)
    # - value container (this is where we hold values and we must not exceed 4096 in total value memory)
    # - serialization function (this take values and turns them into bytes)
    # - deserialization function (look at the bytes in the format mentioned above and turn it into readable text)

    def __init__(self, max_size = 4096):
        self.max_size = max_size
        self.current_size = 0
        self.value_container = []


    def add_value(self, value:bytes):
        # byte_value = bytes(value, 'utf-8') this only need to be here if you you are struggling with trasnforming this later
        value_memory_usage = len(value) + 4 #4 for the prefix and len(byte_value) to get the memory of the value as each string character is 1 byte
        if value_memory_usage + self.current_size > self.max_size:
            raise ValueError("Value too big")
        self.value_container.append(value)
        self.current_size +=value_memory_usage


    def to_bytes(self):
        # get the length of the value (because you need to prefix you data with its length so that we konw how much to read)
        byte_list = bytearray() #make an empty bytearray like you would a list when appending values
        for val in self.value_container:
            prefix = len(val)
            byte_list.extend(prefix.to_bytes(4,'big'))
            byte_list.extend(val)
        return bytes(byte_list) #this must be returned as bytes(byte_list) so that its immutable during travel
    

    def from_bytes(self, raw:bytes):
        page = Page(max_size=4096) #create a new page to hold the disk values
        i = 0 #this is the index which we use to create the offset
        while i < len(raw):
            # look at the first 4 digits and understand what the length of the value is
            length = int.from_bytes(raw[i:i+4], 'big')
            # Extract this many elements from raw data starting at i+4
            value = raw[i+4:prefix]
            # extract this value and save it somewhere
            deserialization_output.append(value)
            # move the index forward by that many spaces
            i += len(value)

            

In [252]:
raw = (
    b'\x00\x00\x00\x05' + b'hello' +
    b'\x00\x00\x00\x06' + b'world!'
)


In [253]:
raw

b'\x00\x00\x00\x05hello\x00\x00\x00\x06world!'

In [254]:
deserialization_list = []
i = 0

while i < len(raw):
    length = int.from_bytes(raw[i:i+4],'big')
    current_encoded_value = raw[i+4:i+4+length]
    deserialization_list.append(current_encoded_value)
    i+=length+4

In [255]:
deserialization_list

[b'hello', b'world!']

In [226]:
page = Page()

In [227]:
page.add_value(bytes('ali','utf-8'))

In [229]:
page.to_bytes

<bound method Page.to_bytes of <__main__.Page object at 0x106482cf0>>

In [230]:
print(int.from_bytes(page.to_bytes,'big'))



TypeError: cannot convert 'method' object to bytes